# LFW 매니페스트 생성

LFW deep-funneled 이미지를 전부 확인한 뒤 identity 단위로 `development / calibration / test`를 분리하고, 다음 세 입력을 생성합니다.

- `face_manifest.csv`: `image_id, identity_id, split, image_path`
- `gallery_identities.txt`: test split에서 등록 인물로 사용할 ID
- `unknown_unknown_identities.txt`: test split의 unknown unknown ID

`MODE`, `DATA_FRACTION`, `SEED`는 첫 설정 셀에서만 바꿉니다. 전체 split을 만든 뒤 development/calibration은 각 split 안에서, test는 registered/known-unknown/unknown-unknown 역할 안에서 identity 단위로 선택합니다. `WRITE_OUTPUTS=False`여도 전체 검증과 선택은 실행하며 파일만 저장하지 않습니다.

## 중단 후 재시작

- 매니페스트 생성 셀 이전/도중에 중단: 커널을 재시작하고 첫 코드 셀부터 다시 실행합니다.
- 저장 셀에서 중단: 출력 폴더를 확인한 뒤 `OVERWRITE=True`로 바꾸고 첫 코드 셀부터 다시 실행합니다. 각 파일은 임시 파일 작성 후 교체됩니다.
- scope가 바뀌면 기존 run과 섞지 말고 새 run을 만드십시오. `real, 1.0`만 전체 데이터 논문 결과입니다.


In [ ]:
from __future__ import annotations

from dataclasses import replace
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'research').is_dir() and (candidate / 'configs').is_dir():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다. D:/ronbun 안에서 노트북을 실행하십시오.')


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.compression import PCA_SWEEP_DIMENSIONS
from research.datasets import build_lfw_manifest, write_lfw_manifest_bundle
from research.experiments.scope import (
    ExperimentScope,
    select_manifest_fraction,
    select_open_set_protocol_fraction,
)
from research.protocols import build_open_set_protocol, validate_identity_disjoint_splits

print(f'project_root: {PROJECT_ROOT}')
print(f'python: {sys.executable}')
print(f'pandas: {pd.__version__}')


## 1. 경로와 분할 조건 고정

`MODE=dev`는 빠른 점검, `MODE=real`은 논문 결과 의도를 나타냅니다. 실제 범위는 `DATA_FRACTION`이 결정하며, `real, 1.0`만 full-data 결과입니다. 기본 `MIN_GALLERY_IMAGES=6`은 enrollment 5장 뒤에도 registered probe가 남도록 하는 조건입니다.


In [ ]:
# Step 1 실행 범위: 이 셀의 세 값만 바꾸고 Kernel Restart -> Run All
MODE = 'real'             # 'dev' 또는 'real'
DATA_FRACTION = 1.0     # 0 < DATA_FRACTION <= 1
SEED = 42

WRITE_OUTPUTS = True  # 검증 완료 후에만 True
OVERWRITE = True      # 기존 결과를 의도적으로 교체할 때만 True

PCA_DIMENSIONS = (384, 256, 128, 64, 32)
PQ_SOURCE_DIMENSION = 512
if PCA_DIMENSIONS != tuple(PCA_SWEEP_DIMENSIONS):
    raise RuntimeError('노트북 PCA sweep과 research.compression 정의가 다릅니다.')
EXPERIMENT_SCOPE = ExperimentScope(
    mode=MODE, data_fraction=DATA_FRACTION, seed=SEED
)

DEVELOPMENT_FRACTION = 0.60
CALIBRATION_FRACTION = 0.20
GALLERY_SIZE = 50
MIN_GALLERY_IMAGES = 6
ENROLLMENT_COUNT = 5
UNKNOWN_UNKNOWN_FRACTION = 0.50

LFW_ROOT = PROJECT_ROOT / 'data/raw/LFW/lfw-deepfunneled/lfw-deepfunneled'
OUTPUT_DIR = PROJECT_ROOT / 'data/interim/lfw'

display(pd.Series({
    'WRITE_OUTPUTS': WRITE_OUTPUTS,
    'OVERWRITE': OVERWRITE,
    'LFW_ROOT': str(LFW_ROOT),
    'OUTPUT_DIR': str(OUTPUT_DIR),
    **EXPERIMENT_SCOPE.as_dict(),
    'SEED': SEED,
    'PCA_DIMENSIONS': PCA_DIMENSIONS,
    'PQ_SOURCE_DIMENSION': PQ_SOURCE_DIMENSION,
    'GALLERY_SIZE': GALLERY_SIZE,
    'MIN_GALLERY_IMAGES': MIN_GALLERY_IMAGES,
}, name='value').to_frame())


## 2. 전체 데이터 검사와 매니페스트 구성

먼저 전체 데이터에서 identity-disjoint split과 open-set 역할을 검증합니다. 그 다음 공통 `research.experiments.scope` API로 development/calibration 및 각 open-set 역할을 독립적으로 축소합니다. 이미지 단위 무작위 절단은 하지 않습니다.


In [ ]:
full_bundle = build_lfw_manifest(
    LFW_ROOT,
    PROJECT_ROOT,
    seed=SEED,
    development_fraction=DEVELOPMENT_FRACTION,
    calibration_fraction=CALIBRATION_FRACTION,
    gallery_size=GALLERY_SIZE,
    min_gallery_images=MIN_GALLERY_IMAGES,
    unknown_unknown_fraction=UNKNOWN_UNKNOWN_FRACTION,
)
full_protocol = build_open_set_protocol(
    full_bundle.manifest,
    full_bundle.gallery_identities,
    full_bundle.unknown_unknown_identities,
    enrollment_count=ENROLLMENT_COUNT,
    seed=SEED,
)
selected_protocol = select_open_set_protocol_fraction(
    full_protocol,
    EXPERIMENT_SCOPE,
    namespace='lfw:test',
)
selected_development = select_manifest_fraction(
    full_bundle.manifest.loc[full_bundle.manifest['split'].eq('development')],
    EXPERIMENT_SCOPE,
    namespace='lfw:development',
)
selected_calibration = select_manifest_fraction(
    full_bundle.manifest.loc[full_bundle.manifest['split'].eq('calibration')],
    EXPERIMENT_SCOPE,
    namespace='lfw:calibration',
)
selected_test = pd.concat(
    [
        selected_protocol.gallery,
        selected_protocol.registered_probes,
        selected_protocol.known_unknown_probes,
        selected_protocol.unknown_unknown_probes,
    ],
    ignore_index=True,
)
selected_image_ids = set(pd.concat(
    [selected_development, selected_calibration, selected_test],
    ignore_index=True,
)['image_id'].astype(str))
selected_manifest = full_bundle.manifest.loc[
    full_bundle.manifest['image_id'].astype(str).isin(selected_image_ids)
].copy().reset_index(drop=True)
validate_identity_disjoint_splits(selected_manifest)
selected_gallery_ids = tuple(sorted(
    selected_protocol.gallery['identity_id'].astype(str).unique()
))
selected_unknown_ids = tuple(sorted(
    selected_protocol.unknown_unknown_probes['identity_id'].astype(str).unique()
))
if len(selected_development) < max(PCA_DIMENSIONS):
    raise ValueError(
        '선택된 development 이미지 수가 PCA-384 학습에 부족합니다. '
        'DATA_FRACTION을 늘리십시오.'
    )
summary = {
    **full_bundle.summary,
    'scope': EXPERIMENT_SCOPE.as_dict(),
    'source_image_count': int(len(full_bundle.manifest)),
    'image_count': int(len(selected_manifest)),
    'identity_count': int(selected_manifest['identity_id'].nunique()),
    'development_identity_count': int(selected_development['identity_id'].nunique()),
    'calibration_identity_count': int(selected_calibration['identity_id'].nunique()),
    'test_identity_count': int(selected_test['identity_id'].nunique()),
    'gallery_identity_count': len(selected_gallery_ids),
    'known_unknown_identity_count': int(
        selected_protocol.known_unknown_probes['identity_id'].nunique()
    ),
    'unknown_unknown_identity_count': len(selected_unknown_ids),
}
bundle = replace(
    full_bundle,
    manifest=selected_manifest,
    gallery_identities=selected_gallery_ids,
    unknown_unknown_identities=selected_unknown_ids,
    summary=summary,
)

display(pd.Series(bundle.summary, name='value').to_frame())
display(
    bundle.manifest.groupby('split').agg(
        images=('image_id', 'size'),
        identities=('identity_id', 'nunique'),
    )
)
display(bundle.manifest.head(10))
print('LFW identity-aware scope 검증 통과')


## 3. 결과 저장

기본값에서는 저장 예정 경로만 보여 줍니다. 실제 파일이 필요하면 위 설정 셀의 `WRITE_OUTPUTS=True`로 변경하고 노트북 전체를 다시 실행하십시오. 기존 파일이 있으면 `OVERWRITE=False`가 덮어쓰기를 막습니다.


In [ ]:
planned_paths = {
    name: OUTPUT_DIR / name
    for name in (
        'face_manifest.csv',
        'gallery_identities.txt',
        'unknown_unknown_identities.txt',
        'summary.json',
    )
}

if WRITE_OUTPUTS:
    written_paths = write_lfw_manifest_bundle(
        bundle, OUTPUT_DIR, overwrite=OVERWRITE
    )
    print('저장 완료')
else:
    written_paths = planned_paths
    print('WRITE_OUTPUTS=False: 검증만 완료했으며 파일은 저장하지 않았습니다.')

display(pd.DataFrame(
    [{'file': name, 'path': str(path), 'exists': path.is_file()}
     for name, path in written_paths.items()]
))


## 다음 단계

저장 후 Step 1 기준 설정 `configs/experiments/step1_embedding_compression.yaml`이 아래 경로를 가리키는지 확인합니다. 기존 00~05 DB 시스템 runbook은 과거 재현용이며 Step 1의 fallback-free 평가 경로와 섞지 않습니다.

```yaml
dataset:
  manifest_path: data/interim/lfw/face_manifest.csv
protocol:
  gallery_identities_path: data/interim/lfw/gallery_identities.txt
  unknown_unknown_identities_path: data/interim/lfw/unknown_unknown_identities.txt
```

생성된 `summary.json`의 `scope`와 실행 config의 `execution` 값이 같아야 합니다. 이 노트북은 설정 파일을 자동 변경하지 않습니다.
